In [0]:

raw_path        = "abfss://raw@nyctaxidata28.dfs.core.windows.net/trip-data/"
bronze_path     = "abfss://bronze@nyctaxidata28.dfs.core.windows.net/taxi_data/"
checkpoint_path = "abfss://bronze@nyctaxidata28.dfs.core.windows.net/checkpoints/taxi/"
schema_path     = "abfss://bronze@nyctaxidata28.dfs.core.windows.net/schema/taxi/"


df = (spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.useManagedFileEvents", "false") \
    .load(raw_path)
)

query = df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(bronze_path)

query.awaitTermination()
print(" Autoloader completed!")



In [0]:

bronze_df = spark.read.format("delta").load(bronze_path)
print("Row count:", bronze_df.count())
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

In [0]:
bronze_df.printSchema()

In [0]:
display(bronze_df)